## 1. Instalare dependențe
Rulare recomandată în Google Colab cu GPU activat.

In [1]:
%pip install -q accelerate transformers datasets[audio] evaluate jiwer soundfile
!apt-get -qq update
!apt-get -qq install ffmpeg

Note: you may need to restart the kernel to use updated packages.


"apt-get" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"apt-get" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


## 2. Montează Google Drive și setează căile
Copiază folderul proiectului în Drive înainte de rulare.

In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/whisper_ro_project')
DATASET_ROOT = PROJECT_ROOT / '1774203787031-cv-corpus-25.0-2026-03-09-ro'
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
os.chdir(PROJECT_ROOT)
PROJECT_ROOT, DATASET_ROOT, ARTIFACTS_ROOT

ModuleNotFoundError: No module named 'google'

## 3. Pregătește manifestele și audio normalizat
Se exportă fișiere CSV cu coloanele `audio_path` și `text`.

In [7]:
from pathlib import Path

import pandas as pd



from asr_ro.data_prep import prepare_all_manifests



summaries = prepare_all_manifests(

    dataset_root=DATASET_ROOT,

    output_root=ARTIFACTS_ROOT / 'cv_ro',

    normalize_audio=True,

    min_duration_ms=500,

    max_duration_ms=30000,

)



manifest_dir = ARTIFACTS_ROOT / 'cv_ro' / 'manifests'

preview_frames = {

    split: pd.read_csv(manifest_dir / f'{split}.csv').head(3)

    for split in ('train', 'dev', 'test')

}



print('Manifeste generate în:', manifest_dir)

for split, frame in preview_frames.items():

    print(f'\n[{split}] coloane:', list(frame.columns))

    display(frame)



summaries


ModuleNotFoundError: No module named 'asr_ro'

## 4. Fine-tuning Whisper base
Parametrii sunt setați conservator pentru un GPU Colab obișnuit.

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    '-m',
    'asr_ro.train_whisper',
    '--train-csv', str(ARTIFACTS_ROOT / 'cv_ro' / 'manifests' / 'train.csv'),
    '--dev-csv', str(ARTIFACTS_ROOT / 'cv_ro' / 'manifests' / 'dev.csv'),
    '--output-dir', str(ARTIFACTS_ROOT / 'whisper-base-ro'),
    '--model-name', 'openai/whisper-base',
    '--train-batch-size', '8',
    '--eval-batch-size', '8',
    '--gradient-accumulation-steps', '2',
    '--num-train-epochs', '8',
    '--learning-rate', '1e-5',
    '--logging-steps', '25',
    '--save-total-limit', '2',
    '--fp16',
    '--freeze-encoder',
]
subprocess.run(command, check=True)

## 5. Evaluare pe `test.tsv`
Se calculează `WER` și `CER` pentru baseline și pentru modelul fine-tuned.

In [ ]:
from asr_ro.evaluate_model import compare_models

comparison = compare_models(
    test_csv=ARTIFACTS_ROOT / 'cv_ro' / 'manifests' / 'test.csv',
    fine_tuned_model=str(ARTIFACTS_ROOT / 'whisper-base-ro'),
    baseline_model='openai/whisper-base',
    output_path=ARTIFACTS_ROOT / 'evaluation' / 'comparison.json',
    limit=None,
    batch_size=4,
)
comparison['baseline_metrics'], comparison['fine_tuned_metrics'], comparison['improvement']

## 6. Vizualizare rezultate
Graficul de mai jos pune în paralel `WER` și `CER` înainte și după fine-tuning.

In [ ]:
import matplotlib.pyplot as plt

baseline = comparison['baseline_metrics']
fine_tuned = comparison['fine_tuned_metrics']
metrics = ['wer', 'cer']
baseline_values = [baseline[m] for m in metrics]
fine_tuned_values = [fine_tuned[m] for m in metrics]

plt.figure(figsize=(6, 4))
x_positions = range(len(metrics))
plt.bar([x - 0.15 for x in x_positions], baseline_values, width=0.3, label='Baseline')
plt.bar([x + 0.15 for x in x_positions], fine_tuned_values, width=0.3, label='Fine-tuned')
plt.xticks(list(x_positions), [m.upper() for m in metrics])
plt.ylabel('Error rate')
plt.title('Whisper base vs Whisper base fine-tuned pe Common Voice ro')
plt.legend()
plt.show()

## 7. Observații pentru raport
- datele folosite provin din split-urile oficiale `train/dev/test`;
- audio a fost resamplat la `16 kHz` mono;
- comparația se face cu `WER` și `CER`;
- verifică exemplele din `comparison['examples']` pentru cazurile în care accentul românesc este mai bine surprins după fine-tuning.